# Итоговое сравнение конкретных моделей — v22 / Run All

Нужна лучшая итоговая модель, не обязательно самый стабильный рецепт. Сравниваем **сохранённые веса**: новый v21, прежний primary equal3, готовый full-train equal3 v18 и MVP. Нового обучения, пересчёта BatchNorm и подбора λ нет.

**Restart Kernel → Run All** в прежнем исследовательском `.venv`. Интернет и другое устройство не нужны. Общего ограничения времени нет. Оставь компьютер на питании и без сна.

Все системы используют одни исходные calibration/validation. Каждая получает свой порог только на calibration; все пороги фиксируются до расчёта validation. MVP также показывается с его действующим историческим порогом. Рабочий MVP, исходная validation/bbox, старые веса и runs не меняются.

Outer уже наблюдалась: это **development-сравнение**, а не независимая проверка на скрытом тесте. Автоматической смены MVP нет.


In [1]:
from pathlib import Path
import os, sys, json, subprocess
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/final_model_comparison.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
from IPython.display import Markdown, display
from training import final_model_comparison as experiment
RUN_NAME = 'comparison_v1'
SOURCE_RUN = 'review_v1'
SEARCH_RUN = 'search_v1'
POLICY_RUN = 'policy_v1'
ALLOW_OUTER_EVALUATION = True  # Согласованное сравнение уже наблюдавшейся outer validation.
source_manifest = json.loads((REPO / 'OSNet-AIN-x1.0/variant_16_review_protocol/runs' / SOURCE_RUN / 'manifest.json').read_text())
torch.set_num_threads(source_manifest['runtime']['torch_threads'])
torch.use_deterministic_algorithms(source_manifest['runtime']['deterministic'])
print('Repository:', REPO, '\nKernel:', sys.executable)
print('Run:', RUN_NAME, '| torch:', source_manifest['runtime']['device'], '| ONNX: CPU | обучение: нет')


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK 
Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Run: comparison_v1 | torch: mps | ONNX: CPU | обучение: нет


## 1. Быстрые проверки
Синтетические тесты не запускают обучение и не используют исходную validation.


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_final_model_comparison.py', 'tests/test_model_search.py'], check=True)


..............................                                           [100%]
30 passed in 3.02s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_final_model_comparison.py', 'tests/test_model_search.py'], returncode=0)

## 2. Источники и замороженный план
Читаются завершённые v16/v18/v21 и сохранённые веса v20. Ни один исходный run не запускается и не переписывается.


In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN, search_run=SEARCH_RUN, policy_run=POLICY_RUN)
print('Результаты:', context['output'])
for spec in context['manifest']['comparison_plan']['systems']:
    print(spec['name'], '| train ID:', spec['train_identities'], '| lambda:', spec['lambda'], '| candidate:', spec['candidate_policy'])
print('Дополнительно: MVP_active с неизменённым действующим порогом')


Результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_22_final_comparison/runs/comparison_v1
V21_selected_primary | train ID: 740 | lambda: 0.65 | candidate: raw_top1
R1_equal3_primary | train ID: 740 | lambda: 0.75 | candidate: raw_top1
R1_equal3_full_v18 | train ID: 925 | lambda: 0.75 | candidate: raw_top1
MVP_recalibrated | train ID: 925 | lambda: 0.5 | candidate: ranking_top1
Дополнительно: MVP_active с неизменённым действующим порогом


## 3. Calibration → фиксация → validation → отчёт
Показываются этап, профиль, количество обработанных изображений и время. Готовые задачи восстанавливаются по checksum после повторного Run All. Ошибка калибровки не допускает перехода к validation.


In [4]:
result = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)



STAGE 1/3: calibration features and thresholds
[START] features_calibration_R1_control_20260915 | elapsed 0.00 h
  calibration/R1_control_20260915: 32/1202 images
  calibration/R1_control_20260915: 352/1202 images
  calibration/R1_control_20260915: 672/1202 images
  calibration/R1_control_20260915: 992/1202 images
  calibration/R1_control_20260915: 1202/1202 images
[DONE] features_calibration_R1_control_20260915 | 0.3 min
[START] features_calibration_R1_control_20260916 | elapsed 0.01 h
  calibration/R1_control_20260916: 32/1202 images
  calibration/R1_control_20260916: 352/1202 images
  calibration/R1_control_20260916: 672/1202 images
  calibration/R1_control_20260916: 992/1202 images
  calibration/R1_control_20260916: 1202/1202 images
[DONE] features_calibration_R1_control_20260916 | 0.3 min
[START] features_calibration_R1_control_20260917 | elapsed 0.01 h
  calibration/R1_control_20260917: 32/1202 images
  calibration/R1_control_20260917: 352/1202 images
  calibration/R1_control_20

In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
print('Полные результаты:', context['output'] / 'results.json')
print('Пороги до validation:', context['output'] / 'frozen_comparison.json')
assert result['status'] == 'complete' and result['protected_unchanged']
assert result['optimizer_updates'] == 0 and result['bn_updates'] == 0 and not result['promoted']


# v22 — сравнение конкретных сохранённых моделей

Статус: **complete**.
Это уже наблюдавшиеся outer development данные, не независимый тест и не оценка hidden test.
Нового обучения, пересчёта BatchNorm, поиска λ и смены MVP нет. Пороги выбраны только на calibration.

| Система | train ID | λ | Порог | mAP@10 | Rank-1 | Rank-5 | F1 | TNR | C | Принято / отказ |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| V21_selected_primary | 740 | 0.65 | 0.52370554 | 0.810805 | 0.805668 | 0.878543 | 0.800000 | 0.677419 | 0.763226 | 225 / 84 |
| R1_equal3_primary | 740 | 0.75 | 0.51207143 | 0.808286 | 0.805668 | 0.882591 | 0.799107 | 0.645161 | 0.752923 | 230 / 79 |
| R1_equal3_full_v18 | 925 | 0.75 | 0.53436518 | 0.805169 | 0.813765 | 0.878543 | 0.795455 | 0.709677 | 0.769721 | 218 / 91 |
| MVP_recalibrated | 925 | 0.50 | 0.59487545 | 0.814689 | 0.801619 | 0.882591 | 0.728606 | 0.790323 | 0.747121 | 187 / 122 |
| MVP_active | 925 | 0.50 | 0.59487545 | 0.814689 | 0.801619 | 0.882591 | 0.728606 | 0.790323 | 0.747121 | 187 / 122 |

Новый v21 против R1_equal3_primary: ΔmAP +0.252 п.п.; ΔC +1.030 п.п.; запросов лучше/хуже/без изменений: 20/21/206.

Новый v21 против R1_equal3_full_v18: ΔmAP +0.564 п.п.; ΔC -0.650 п.п.; запросов лучше/хуже/без изменений: 29/25/193.

Новый v21 против MVP_recalibrated: ΔmAP -0.388 п.п.; ΔC +1.610 п.п.; запросов лучше/хуже/без изменений: 34/34/179.

Новый v21 против MVP_active: ΔmAP -0.388 п.п.; ΔC +1.610 п.п.; запросов лучше/хуже/без изменений: 34/34/179.

## Как читать результат

- V21_selected_primary и R1_equal3_primary обучались на 740 ID; full_v18 и MVP — на 925. Это разные конкретные веса, не чистая абляция рецепта.
- MVP_active использует действующий исторический порог. MVP_recalibrated — отдельная оценка того же encoder с общей calibration-only процедурой; рабочий порог не меняется.
- C = 0.7 F1 + 0.3 TNR. Ранжирование и выбор принятого кандидата оцениваются отдельно; максимум mAP не обязательно максимум C.
- raw_map и попарные AP сохранены в results.json как диагностика, без дополнительного отбора конфигураций.
- Метрики получены из новых CSV официальным evaluator: десять ID даже при отказе. Junk удаляется evaluator уже после подачи top-10; одиннадцатое место не восстанавливается.
- Исторические числа не подставляются вместо нового расчёта. Сравнение не является GPU benchmark или приёмкой релиза.
- Экспорты — для исходного validation-протокола, не файлы для отправки на неизвестный test. Порядок реальных float32-векторов: query, затем gallery (embedding_order.json).
- Готовые задачи возобновляются по checksum. Для изменения кода нужен новый RUN_NAME. Общего ограничения времени нет.
- Исходные файлы сохранены: True. Автоматическое продвижение: False.
- Время текущего запуска: 7.3 мин; cached-задачи не являются новым замером.


Полные результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_22_final_comparison/runs/comparison_v1/results.json
Пороги до validation: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_22_final_comparison/runs/comparison_v1/frozen_comparison.json
